# Load And Merge All Markets Data

这个 notebook 用统一 schema 加载并合并 A 股、港股、美股数据。

设计约定：
- 跨市场主键使用 `instrument_id = market + ':' + symbol`
- `daily_latest` 和 `history_2y` 使用同一套日线字段，方便直接拼接或对比
- 文件级元数据额外保留在 `file_trade_date` / `history_end_date` / `file_path`


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'data_store').exists():
        return cwd
    if cwd.name == 'notebooks' and (cwd.parent / 'data_store').exists():
        return cwd.parent
    if (cwd.parent / 'data_store').exists():
        return cwd.parent
    raise FileNotFoundError('Could not locate project root containing data_store/')


PROJECT_ROOT = resolve_project_root()
DATA_ROOT = PROJECT_ROOT / 'data_store'
LATEST_ROOT = DATA_ROOT / 'daily_latest'
HISTORY_ROOT = DATA_ROOT / 'history_2y'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('LATEST_ROOT  =', LATEST_ROOT)
print('HISTORY_ROOT =', HISTORY_ROOT)


PROJECT_ROOT = /Users/yugh/YUGH/dev/trade
LATEST_ROOT  = /Users/yugh/YUGH/dev/trade/data_store/daily_latest
HISTORY_ROOT = /Users/yugh/YUGH/dev/trade/data_store/history_2y


## Unified Data Format

核心字段：
- `market`: `a` / `hk` / `us`
- `symbol`: 市场内原始股票代码
- `provider_symbol`: 对应数据源请求代码
- `trade_date`: 行情日期
- `open/high/low/close/volume/amount/turnover_rate/outstanding_share`
- `source`: `sina` 或 `sina_snapshot`

扩展字段：
- `instrument_id`: 跨市场唯一键，例如 `a:000001`、`hk:00700`、`us:AAPL`
- `file_trade_date`: latest 文件名里的日期
- `history_end_date`: history 文件名里的回补截止日
- `file_path`: 原始文件路径，方便追溯来源

注意：
- `daily_latest` 是当日横截面，不是完整历史
- `history_2y` 是按 ticker 存的，需要批量读取后再跨市场合并
- 全量历史文件可能很大，示例里默认只加载每个市场前几只股票


In [2]:
LATEST_COLUMNS = [
    'market', 'symbol', 'provider_symbol', 'trade_date', 'open', 'high', 'low', 'close',
    'volume', 'amount', 'turnover_rate', 'outstanding_share', 'source',
]


def _read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(
        path,
        dtype={
            'market': 'string',
            'symbol': 'string',
            'provider_symbol': 'string',
            'source': 'string',
        },
    )


def _canonicalize_daily(frame: pd.DataFrame) -> pd.DataFrame:
    df = frame.copy()
    df['market'] = df['market'].astype('string')
    df['symbol'] = df['symbol'].astype('string')
    df['provider_symbol'] = df['provider_symbol'].astype('string')
    df['instrument_id'] = df['market'] + ':' + df['symbol']
    df['trade_date'] = pd.to_datetime(df['trade_date'], errors='coerce')
    for column in ['open', 'high', 'low', 'close', 'volume', 'amount', 'turnover_rate', 'outstanding_share']:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors='coerce')
    return df


def load_latest_all() -> pd.DataFrame:
    frames = []
    for path in sorted(LATEST_ROOT.glob('*.csv')):
        market, file_trade_date = path.stem.split('.', 1)
        frame = _canonicalize_daily(_read_csv(path))
        frame['file_trade_date'] = pd.to_datetime(file_trade_date, errors='coerce')
        frame['file_path'] = str(path.relative_to(PROJECT_ROOT))
        frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=LATEST_COLUMNS + ['instrument_id', 'file_trade_date', 'file_path'])
    return pd.concat(frames, ignore_index=True)


def load_history_all(markets=('a', 'hk', 'us'), max_files_per_market=None) -> pd.DataFrame:
    frames = []
    for market in markets:
        market_root = HISTORY_ROOT / market
        if not market_root.exists():
            continue
        paths = sorted(market_root.glob('*.history_2y.csv'))
        if max_files_per_market is not None:
            paths = paths[:max_files_per_market]
        for path in paths:
            stem = path.name.replace('.history_2y.csv', '')
            symbol, history_end_date = stem.rsplit('.', 1)
            frame = _canonicalize_daily(_read_csv(path))
            frame['history_end_date'] = pd.to_datetime(history_end_date, errors='coerce')
            frame['file_symbol'] = symbol
            frame['file_path'] = str(path.relative_to(PROJECT_ROOT))
            frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=LATEST_COLUMNS + ['instrument_id', 'history_end_date', 'file_symbol', 'file_path'])
    return pd.concat(frames, ignore_index=True)


In [3]:
latest_all = load_latest_all()

display(latest_all.groupby('market').agg(rows=('instrument_id', 'size'), tickers=('instrument_id', 'nunique')))
display(latest_all[['instrument_id', 'provider_symbol', 'trade_date', 'close', 'volume', 'source']].head(10))


,rows,tickers
market,,
a,5498,5498
hk,2740,2740
us,17139,17138


,instrument_id,provider_symbol,trade_date,close,volume,source
0,a:1,sz000001,2026-04-09,11.10,60236493.0,sina_snapshot
1,a:2,sz000002,2026-04-09,3.87,85780318.0,sina_snapshot
2,a:4,sz000004,2026-04-09,3.35,12547962.0,sina_snapshot
3,a:6,sz000006,2026-04-09,8.52,17158811.0,sina_snapshot
4,a:7,sz000007,2026-04-09,13.40,9498055.0,sina_snapshot
5,a:8,sz000008,2026-04-09,2.74,143854035.0,sina_snapshot
6,a:9,sz000009,2026-04-09,8.67,18097245.0,sina_snapshot
7,a:10,sz000010,2026-04-09,3.71,21822100.0,sina_snapshot
8,a:11,sz000011,2026-04-09,8.36,5453610.0,sina_snapshot
9,a:12,sz000012,2026-04-09,4.43,14969123.0,sina_snapshot


In [4]:
latest_sample = (
    latest_all
    .sort_values(['market', 'symbol'])
    .groupby('market', group_keys=False)
    .head(3)
)

display(latest_sample[['instrument_id', 'provider_symbol', 'trade_date', 'open', 'high', 'low', 'close', 'volume']])


,instrument_id,provider_symbol,trade_date,open,high,low,close,volume
0,a:1,sz000001,2026-04-09,11.17,11.22,11.06,11.10,60236493.0
7,a:10,sz000010,2026-04-09,3.84,3.84,3.69,3.71,21822100.0
53,a:100,sz000100,2026-04-09,4.26,4.30,4.24,4.29,258618253.0
5498,hk:00001,00001,2026-04-09,62.80,63.10,62.05,63.00,5319402.0
5499,hk:00002,00002,2026-04-09,74.40,74.55,73.25,74.55,3159700.0
5500,hk:00003,00003,2026-04-09,7.26,7.32,7.16,7.32,25764162.0
8238,us:A,A,2026-04-08,0.00,0.00,0.00,116.92,0.0
8239,us:AA,AA,2026-04-08,0.00,0.00,0.00,71.76,0.0
8240,us:AAA,AAA,2026-04-08,0.00,0.00,0.00,24.91,0.0


In [5]:
# 直接加载三市场当前已落盘的全量历史数据。
history_sample = load_history_all(max_files_per_market=None)

display(history_sample.groupby('market').agg(rows=('instrument_id', 'size'), tickers=('instrument_id', 'nunique'), start=('trade_date', 'min'), end=('trade_date', 'max')))
display(history_sample[['instrument_id', 'trade_date', 'close', 'volume', 'history_end_date', 'file_path']].head(12))


,rows,tickers,start,end
market,,,,
a,77483,160,2024-04-09,2026-04-09
hk,58495,150,2024-04-09,2026-04-09
us,77403,229,2024-04-08,2026-04-08


,instrument_id,trade_date,close,volume,history_end_date,file_path
0,a:600000,2024-04-09,7.26,20335543.0,2026-04-09,data_store/history_2y/a/600000.2026-04-09.hist...
1,a:600000,2024-04-10,7.27,22436726.0,2026-04-09,data_store/history_2y/a/600000.2026-04-09.hist...
2,a:600000,2024-04-11,7.24,25364241.0,2026-04-09,data_store/history_2y/a/600000.2026-04-09.hist...
3,a:600000,2024-04-12,7.13,32809731.0,2026-04-09,data_store/history_2y/a/600000.2026-04-09.hist...
4,a:600000,2024-04-15,7.30,48310304.0,2026-04-09,data_store/history_2y/a/600000.2026-04-09.hist...
5,a:600000,2024-04-16,7.26,40203325.0,2026-04-09,data_store/history_2y/a/600000.2026-04-09.hist...
6,a:600000,2024-04-17,7.34,43151857.0,2026-04-09,data_store/history_2y/a/600000.2026-04-09.hist...
7,a:600000,2024-04-18,7.33,46822476.0,2026-04-09,data_store/history_2y/a/600000.2026-04-09.hist...
8,a:600000,2024-04-19,7.34,38004160.0,2026-04-09,data_store/history_2y/a/600000.2026-04-09.hist...
9,a:600000,2024-04-22,7.29,32694690.0,2026-04-09,data_store/history_2y/a/600000.2026-04-09.hist...


In [6]:
history_last = (
    history_sample
    .sort_values(['instrument_id', 'trade_date'])
    .groupby('instrument_id', group_keys=False)
    .tail(1)[['instrument_id', 'trade_date', 'close']]
)

latest_vs_history = latest_all.merge(
    history_last.rename(columns={'trade_date': '_history_last_trade_date', 'close': '_history_last_close'}),
    on='instrument_id',
    how='left',
)

# 中间校验字段仅用于 notebook 内部比对，最终展示时移除。
latest_output = latest_vs_history.drop(columns=['_history_last_trade_date', '_history_last_close'], errors='ignore')

display(latest_output.head(12))


,market,symbol,provider_symbol,trade_date,open,high,low,close,volume,amount,turnover_rate,outstanding_share,source,instrument_id,file_trade_date,file_path,history_last_trade_date,history_last_close
0,a,1,sz000001,2026-04-09,11.17,11.22,11.06,11.10,60236493.0,669239285.0,NaN,NaN,sina_snapshot,a:1,2026-04-09,data_store/daily_latest/a.2026-04-09.csv,NaT,NaN
1,a,2,sz000002,2026-04-09,3.90,3.91,3.85,3.87,85780318.0,332161942.0,NaN,NaN,sina_snapshot,a:2,2026-04-09,data_store/daily_latest/a.2026-04-09.csv,NaT,NaN
2,a,4,sz000004,2026-04-09,3.35,3.46,3.35,3.35,12547962.0,42099759.0,NaN,NaN,sina_snapshot,a:4,2026-04-09,data_store/daily_latest/a.2026-04-09.csv,NaT,NaN
3,a,6,sz000006,2026-04-09,8.58,8.67,8.37,8.52,17158811.0,146037144.0,NaN,NaN,sina_snapshot,a:6,2026-04-09,data_store/daily_latest/a.2026-04-09.csv,NaT,NaN
4,a,7,sz000007,2026-04-09,12.72,13.53,12.50,13.40,9498055.0,125546485.0,NaN,NaN,sina_snapshot,a:7,2026-04-09,data_store/daily_latest/a.2026-04-09.csv,NaT,NaN
5,a,8,sz000008,2026-04-09,2.81,2.83,2.74,2.74,143854035.0,398177853.0,NaN,NaN,sina_snapshot,a:8,2026-04-09,data_store/daily_latest/a.2026-04-09.csv,NaT,NaN
6,a,9,sz000009,2026-04-09,8.79,8.79,8.63,8.67,18097245.0,157366550.0,NaN,NaN,sina_snapshot,a:9,2026-04-09,data_store/daily_latest/a.2026-04-09.csv,NaT,NaN
7,a,10,sz000010,2026-04-09,3.84,3.84,3.69,3.71,21822100.0,81688016.0,NaN,NaN,sina_snapshot,a:10,2026-04-09,data_store/daily_latest/a.2026-04-09.csv,NaT,NaN
8,a,11,sz000011,2026-04-09,8.51,8.64,8.31,8.36,5453610.0,46138838.0,NaN,NaN,sina_snapshot,a:11,2026-04-09,data_store/daily_latest/a.2026-04-09.csv,NaT,NaN
9,a,12,sz000012,2026-04-09,4.45,4.47,4.42,4.43,14969123.0,66437202.0,NaN,NaN,sina_snapshot,a:12,2026-04-09,data_store/daily_latest/a.2026-04-09.csv,NaT,NaN
